[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/38.%20The%20Automation%20Investment%20Analysis%20Problem/P38-Tier-1_executed.ipynb)

---


# 38. The Automation Investment Analysis Problem
## Tier 1: The Pen-and-Paper Method (Mathematical Formulation)

### Goal and inputs
Choose a portfolio of automation projects that maximizes expected net present value (NPV), subject to a total investment budget. Project benefits depend on which market scenario occurs, and compatible projects can create additional benefits when selected together.

The example has three projects, three market scenarios, a ten-year planning horizon, a 10% annual discount rate, a $50 million budget, and one synergy between Projects 1 and 2. Monetary values are in millions of dollars. Scenario probabilities are 0.4 for High, 0.4 for Base, and 0.2 for Low.

### Modeling assumptions and scope
- Each project is either selected in full or not selected; partial investment is not allowed.
- The market scenario is uncertain when the portfolio is chosen. The same portfolio is evaluated under every scenario, and the objective uses probability-weighted NPV. Project-selection decisions are therefore made before the scenario is known.
- A selected project earns its scenario-specific annual benefit in every year of the planning horizon, beginning in year 1. The model does not choose a start date or represent a gradual implementation schedule.
- Project costs are paid once at the beginning of the planning horizon. Annual benefits and synergy benefits are discounted at the stated rate.
- The synergy adds a percentage of the two selected projects' combined annual benefits in each year. It does not change investment costs.
- The model excludes taxes, financing, operating costs, salvage values, and dependencies or incompatibilities other than the stated synergy.

### Example data
| Project | Initial investment ($M) | High annual benefit ($M/year) | Base annual benefit ($M/year) | Low annual benefit ($M/year) |
|---|---:|---:|---:|---:|
| Autonomous Crane System | 20 | 8 | 5 | 2 |
| AI-Powered Yard Management | 15 | 6 | 4 | 2 |
| Integrated Digital Twin Platform | 18 | 7 | 4 | 1 |

Projects 1 and 2 receive a 15% synergy bonus on their combined annual benefits when both are selected. The budget limits total initial investment to $50 million.

### Solution sequence
1. Represent projects, scenarios, and synergies as data.
2. Calculate standalone and portfolio NPVs to make the financial quantities explicit.
3. Formulate a mixed-integer linear program (MILP) for expected NPV maximization.
4. Solve the model with PuLP's CBC solver and inspect the selected portfolio.
5. Vary the budget in a sensitivity analysis while keeping the example's other inputs fixed.

The MILP is the optimization method used throughout this notebook. Scenario analysis represents uncertainty; it does not introduce scenario-dependent project-selection decisions.

## Mathematical model

The model chooses a portfolio of projects and accounts for the extra value created by selected synergistic pairs. A binary decision is a yes-or-no choice: it equals 1 when the choice is made and 0 otherwise. A continuous decision can take any value within its specified bounds. These variable types represent discrete project choices while keeping the yearly benefit-activation variables explicit.

Let $I$ be the set of projects, $S$ the set of scenarios, $G$ the set of synergy relationships, and $T=\{1,\ldots,H\}$ the years in the planning horizon. The model maximizes expected NPV:

$$
\max \quad
\sum_{s\in S} p_s \left[
\sum_{i\in I}\sum_{t\in T}
\frac{b_{is}}{(1+r)^t}y_{it}
- \sum_{i\in I} c_i x_i
+ \sum_{g\in G}\sum_{t\in T}
\frac{\alpha_g(b_{i_gs}+b_{j_gs})}{(1+r)^t}z_g
\right].
$$

Here, $p_s$ is the probability of scenario $s$, $b_{is}$ is project $i$'s annual benefit in scenario $s$, $c_i$ is its initial cost, $r$ is the annual discount rate, and $H$ is the number of years. Synergy relationship $g$ joins projects $i_g$ and $j_g$ and has bonus percentage $\alpha_g$. The binary variable $x_i$ equals 1 when project $i$ is selected. The variable $y_{it}$ represents whether its benefits are active in year $t$; this example requires $y_{it}=x_i$ in every year. The binary variable $z_g$ equals 1 when both projects in synergy relationship $g$ are selected. Each discounted benefit is expressed in millions of dollars at the beginning of the planning horizon.

The scenario probabilities sum to 1, so weighting the initial cost by scenario probability in the displayed objective is equivalent to subtracting each selected project's cost once. The objective therefore evaluates one common portfolio across all scenarios and maximizes its probability-weighted NPV.

The constraints are:

$$
\begin{aligned}
&\sum_{i\in I} c_i x_i \le B,
&&\text{budget limit},\\
&y_{it} \le x_i, \qquad y_{it}=x_i,
&&i\in I,\ t\in T,\\
&z_g \le x_{i_g}, \qquad z_g \le x_{j_g},
&&g\in G,\\
&z_g \ge x_{i_g}+x_{j_g}-1,
&&g\in G,\\
&x_i\in\{0,1\},\quad z_g\in\{0,1\},\quad 0\le y_{it}\le 1.
&&
\end{aligned}
$$

$B$ is the investment budget. The equality $y_{it}=x_i$ states that a selected project is active in every year; the accompanying inequality makes the implication from selection to activation explicit. With the stated bounds, each $y_{it}$ takes the same zero-or-one value as $x_i$ even though $y_{it}$ is declared continuous. The three inequalities for $z_g$ make it equal to the logical AND of the two project-selection variables: it is 1 exactly when both projects are selected. This linearization is necessary because multiplying two decision variables directly would make the optimization model nonlinear.

A mixed-integer linear program has a linear objective and linear constraints, with at least some decisions restricted to integer values. Here, the project and synergy decisions make the feasible choices combinatorial: the solver must identify the best permissible combination while satisfying the budget and linking constraints. The code uses `x` for project selection, `y` for annual benefit activation, and `z` for synergy selection. The objective and constraints are built in `solve_investment_optimization`.

With $|I|$ projects, $|G|$ synergy relationships, and $H$ years, the model declares $|I|+|G|$ binary variables and $|I|H$ continuous variables. MILP solvers can require exponential time in the worst case as the number of binary decisions grows, although small instances such as this one are suitable for exact optimization.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pulp
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported for automation investment analysis.")

In [ ]:
@dataclass
class AutomationProject:
    """Store a project's investment cost and scenario-specific annual benefits."""

    id: int
    name: str
    cost: float
    benefits: Dict[str, float]


@dataclass
class MarketScenario:
    """Store a market scenario and its probability."""

    name: str
    probability: float


@dataclass
class Synergy:
    """Store a benefit bonus that applies when two projects are both selected."""

    project1_id: int
    project2_id: int
    bonus_percentage: float


print("Project, scenario, and synergy data structures are ready.")

In [ ]:
projects = [
    AutomationProject(
        id=1,
        name="Autonomous Crane System",
        cost=20.0,
        benefits={"High": 8.0, "Base": 5.0, "Low": 2.0},
    ),
    AutomationProject(
        id=2,
        name="AI-Powered Yard Management",
        cost=15.0,
        benefits={"High": 6.0, "Base": 4.0, "Low": 2.0},
    ),
    AutomationProject(
        id=3,
        name="Integrated Digital Twin Platform",
        cost=18.0,
        benefits={"High": 7.0, "Base": 4.0, "Low": 1.0},
    ),
]

scenarios = [
    MarketScenario("High", 0.4),
    MarketScenario("Base", 0.4),
    MarketScenario("Low", 0.2),
]

synergies = [Synergy(1, 2, 0.15)]

budget = 50.0
discount_rate = 0.10
planning_horizon = 10

assert np.isclose(sum(scenario.probability for scenario in scenarios), 1.0)
assert all(project.cost >= 0 for project in projects)
assert all(0 <= scenario.probability <= 1 for scenario in scenarios)

print(
    f"Defined {len(projects)} projects, {len(scenarios)} scenarios, "
    f"and {len(synergies)} synergy relationship."
)
print(f"Budget: ${budget:.1f}M; planning horizon: {planning_horizon} years.")

## NPV calculations

NPV converts cash flows occurring in different years into present-value terms. A benefit received later is worth less than the same benefit received today when the discount rate is positive. For a project with an initial cost paid at year 0 and a constant annual benefit, the scenario-specific NPV is

$$
\operatorname{NPV}_{is}
=-c_i+\sum_{t=1}^{H}\frac{b_{is}}{(1+r)^t}.
$$

In this expression, $i$ identifies a project, $s$ identifies a market scenario, $c_i$ is the initial investment, $b_{is}$ is the annual benefit in that scenario, $r$ is the discount rate, and $H$ is the planning horizon. The code computes the same sum year by year in `calculate_npv`.

The portfolio calculation adds the NPVs of selected projects and, when both members of a synergy pair are selected, discounts that pair's annual bonus over the same horizon. It then weights each scenario result by its probability. These calculations provide a transparent reference for checking the optimization model.

In [ ]:
def calculate_npv(
    project: AutomationProject,
    scenario: MarketScenario,
    discount_rate: float,
    horizon: int,
) -> float:
    """Calculate one project's NPV in one scenario."""
    annual_benefit = project.benefits[scenario.name]
    npv = -project.cost

    for year in range(1, horizon + 1):
        npv += annual_benefit / ((1 + discount_rate) ** year)

    return npv


def calculate_portfolio_npv(
    selected_projects: List[AutomationProject],
    scenarios: List[MarketScenario],
    synergies: List[Synergy],
    discount_rate: float,
    horizon: int,
) -> float:
    """Calculate the probability-weighted NPV of a selected portfolio."""
    expected_npv = 0.0
    selected_by_id = {project.id: project for project in selected_projects}

    for scenario in scenarios:
        scenario_npv = sum(
            calculate_npv(project, scenario, discount_rate, horizon)
            for project in selected_projects
        )

        for synergy in synergies:
            if (
                synergy.project1_id in selected_by_id
                and synergy.project2_id in selected_by_id
            ):
                project1 = selected_by_id[synergy.project1_id]
                project2 = selected_by_id[synergy.project2_id]
                annual_bonus = synergy.bonus_percentage * (
                    project1.benefits[scenario.name]
                    + project2.benefits[scenario.name]
                )

                for year in range(1, horizon + 1):
                    scenario_npv += annual_bonus / (
                        (1 + discount_rate) ** year
                    )

        expected_npv += scenario.probability * scenario_npv

    return expected_npv


print("NPV calculation functions are ready.")

In [ ]:
print("Individual project analysis:")
print("=" * 50)

for project in projects:
    print(f"\n{project.name} (${project.cost:.1f}M):")
    for scenario in scenarios:
        npv = calculate_npv(
            project, scenario, discount_rate, planning_horizon
        )
        print(f"  {scenario.name} scenario NPV: ${npv:.2f}M")

    expected_npv = sum(
        scenario.probability
        * calculate_npv(
            project, scenario, discount_rate, planning_horizon
        )
        for scenario in scenarios
    )
    print(f"  Expected NPV: ${expected_npv:.2f}M")

## Mixed-integer optimization

The model uses binary project-selection variables, annual activation variables, and binary synergy variables. The annual variables preserve the explicit year-by-year benefit calculation; setting them equal to the selection variable applies the assumption that a selected project is active throughout the horizon. The synergy variables and their linking constraints keep the model linear while ensuring the bonus is earned only when both related projects are chosen.

The solver searches the feasible choices defined by the budget and linking constraints and returns a portfolio with the greatest expected NPV. PuLP's `LpVariable` interface accepts the variable category by its `cat` keyword; the project and synergy decisions use `pulp.LpBinary`, while annual activation variables use their specified continuous bounds.

In [ ]:
def solve_investment_optimization(
    projects: List[AutomationProject],
    scenarios: List[MarketScenario],
    synergies: List[Synergy],
    budget: float,
    discount_rate: float,
    horizon: int,
) -> Tuple[pulp.LpProblem, List[AutomationProject]]:
    """Build and solve the expected-NPV investment MILP."""
    project_by_id = {project.id: project for project in projects}
    if len(project_by_id) != len(projects):
        raise ValueError("Project IDs must be unique.")
    if not np.isclose(sum(s.probability for s in scenarios), 1.0):
        raise ValueError("Scenario probabilities must sum to 1.")
    if any(
        synergy.project1_id not in project_by_id
        or synergy.project2_id not in project_by_id
        for synergy in synergies
    ):
        raise ValueError("Each synergy must refer to defined projects.")

    prob = pulp.LpProblem(
        "Automation_Investment_Optimization", pulp.LpMaximize
    )

    # Binary variables encode the portfolio choices made before the scenario is known.
    x = {
        project.id: pulp.LpVariable(
            f"x_{project.id}", lowBound=0, upBound=1, cat=pulp.LpBinary
        )
        for project in projects
    }

    # These variables expose the year-by-year discounted benefit calculation.
    y = {
        (project.id, year): pulp.LpVariable(
            f"y_{project.id}_{year}", lowBound=0, upBound=1
        )
        for project in projects
        for year in range(1, horizon + 1)
    }

    # A synergy variable represents the logical AND of its two project choices.
    z = {
        index: pulp.LpVariable(
            f"z_{index}", lowBound=0, upBound=1, cat=pulp.LpBinary
        )
        for index, _ in enumerate(synergies)
    }

    objective = pulp.lpSum(
        scenario.probability
        * project.benefits[scenario.name]
        / ((1 + discount_rate) ** year)
        * y[project.id, year]
        for scenario in scenarios
        for project in projects
        for year in range(1, horizon + 1)
    )

    objective -= pulp.lpSum(
        project.cost * x[project.id] for project in projects
    )

    objective += pulp.lpSum(
        scenario.probability
        * synergy.bonus_percentage
        * (
            project_by_id[synergy.project1_id].benefits[scenario.name]
            + project_by_id[synergy.project2_id].benefits[scenario.name]
        )
        / ((1 + discount_rate) ** year)
        * z[index]
        for scenario in scenarios
        for index, synergy in enumerate(synergies)
        for year in range(1, horizon + 1)
    )

    prob += objective

    prob += (
        pulp.lpSum(project.cost * x[project.id] for project in projects)
        <= budget,
        "Budget_limit",
    )

    for project in projects:
        for year in range(1, horizon + 1):
            prob += y[project.id, year] <= x[project.id]
            prob += y[project.id, year] == x[project.id]

    for index, synergy in enumerate(synergies):
        project1 = x[synergy.project1_id]
        project2 = x[synergy.project2_id]
        prob += z[index] <= project1
        prob += z[index] <= project2
        prob += z[index] >= project1 + project2 - 1

    print("Solving the investment optimization model...")
    prob.solve(pulp.PULP_CBC_CMD(msg=False))

    if pulp.LpStatus[prob.status] != "Optimal":
        raise RuntimeError(
            f"The solver did not prove optimality: {pulp.LpStatus[prob.status]}"
        )

    selected_projects = [
        project
        for project in projects
        if pulp.value(x[project.id]) > 0.5
    ]

    return prob, selected_projects


print("The investment optimization function is ready.")

In [ ]:
prob, selected_projects = solve_investment_optimization(
    projects,
    scenarios,
    synergies,
    budget,
    discount_rate,
    planning_horizon,
)

print("\n" + "=" * 60)
print("OPTIMIZATION RESULTS")
print("=" * 60)
print(f"\nProblem status: {pulp.LpStatus[prob.status]}")
print(f"Objective value (expected NPV): ${pulp.value(prob.objective):.2f}M")

total_cost = sum(project.cost for project in selected_projects)
portfolio_npv = calculate_portfolio_npv(
    selected_projects,
    scenarios,
    synergies,
    discount_rate,
    planning_horizon,
)

assert total_cost <= budget + 1e-8
assert np.isclose(
    pulp.value(prob.objective), portfolio_npv, rtol=1e-8, atol=1e-8
)

print(f"\nSelected projects ({len(selected_projects)}):")
for project in selected_projects:
    print(f"  - {project.name}: ${project.cost:.1f}M")

print(f"\nTotal investment: ${total_cost:.2f}M")
print(f"Budget utilization: {total_cost / budget * 100:.1f}%")
print(f"Remaining budget: ${budget - total_cost:.2f}M")
print(f"Portfolio expected NPV: ${portfolio_npv:.2f}M")

selected_ids = {project.id for project in selected_projects}
print("\nSynergy effects in the selected portfolio:")
for synergy in synergies:
    if (
        synergy.project1_id in selected_ids
        and synergy.project2_id in selected_ids
    ):
        project1 = next(
            project for project in projects
            if project.id == synergy.project1_id
        )
        project2 = next(
            project for project in projects
            if project.id == synergy.project2_id
        )
        print(
            f"  - {project1.name} + {project2.name}: "
            f"{synergy.bonus_percentage:.0%} of combined annual benefits"
        )

## Budget sensitivity and scenario outcomes

Sensitivity analysis repeats the same optimization model at different budget limits. It shows how the selected portfolio and its expected NPV respond to a changed budget; all project costs, benefits, scenario probabilities, synergies, and discounting assumptions remain fixed. This is an analysis of the same MILP, not a different solution method.

The scenario chart reports the baseline portfolio's NPV under each possible scenario. Each bar is a conditional outcome for that scenario; the probability-weighted expected NPV is the quantity optimized by the model. Scenario NPVs can be negative, so a bar chart is used to display both positive and negative values.

In [ ]:
budget_range = np.arange(30, 81, 5)
budget_results = []

for test_budget in budget_range:
    _, test_selected = solve_investment_optimization(
        projects,
        scenarios,
        synergies,
        float(test_budget),
        discount_rate,
        planning_horizon,
    )

    test_total_cost = sum(project.cost for project in test_selected)
    test_portfolio_npv = calculate_portfolio_npv(
        test_selected,
        scenarios,
        synergies,
        discount_rate,
        planning_horizon,
    )

    budget_results.append(
        {
            "budget": float(test_budget),
            "projects": len(test_selected),
            "total_cost": test_total_cost,
            "npv": test_portfolio_npv,
        }
    )

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
    2, 2, figsize=(15, 12)
)

budgets = [result["budget"] for result in budget_results]
npvs = [result["npv"] for result in budget_results]
ax1.plot(budgets, npvs, "b-o", linewidth=2, markersize=7)
ax1.set_xlabel("Budget ($M)")
ax1.set_ylabel("Expected NPV ($M)")
ax1.set_title("Budget vs. Expected Portfolio NPV")
ax1.grid(True, alpha=0.3)

project_counts = [result["projects"] for result in budget_results]
ax2.bar(budgets, project_counts, color="skyblue", alpha=0.8)
ax2.set_xlabel("Budget ($M)")
ax2.set_ylabel("Number of selected projects")
ax2.set_title("Budget vs. Number of Selected Projects")
ax2.grid(True, axis="y", alpha=0.3)

project_names = [project.name for project in projects]
project_costs = [project.cost for project in projects]
ax3.barh(project_names, project_costs, color="#4ECDC4")
ax3.set_xlabel("Initial investment cost ($M)")
ax3.set_title("Individual Project Costs")
ax3.grid(True, axis="x", alpha=0.3)

scenario_npvs = []
for scenario in scenarios:
    scenario_npv = sum(
        calculate_npv(
            project, scenario, discount_rate, planning_horizon
        )
        for project in selected_projects
    )

    for synergy in synergies:
        if (
            synergy.project1_id in selected_ids
            and synergy.project2_id in selected_ids
        ):
            project1 = next(
                project for project in projects
                if project.id == synergy.project1_id
            )
            project2 = next(
                project for project in projects
                if project.id == synergy.project2_id
            )
            annual_bonus = synergy.bonus_percentage * (
                project1.benefits[scenario.name]
                + project2.benefits[scenario.name]
            )
            for year in range(1, planning_horizon + 1):
                scenario_npv += annual_bonus / (
                    (1 + discount_rate) ** year
                )

    scenario_npvs.append(scenario_npv)

scenario_labels = [
    f"{scenario.name}\n(p={scenario.probability:.0%})"
    for scenario in scenarios
]
ax4.bar(scenario_labels, scenario_npvs, color="#45B7D1")
ax4.axhline(0, color="black", linewidth=0.8)
ax4.set_xlabel("Market scenario and probability")
ax4.set_ylabel("Portfolio NPV ($M)")
ax4.set_title("Baseline Portfolio NPV by Scenario")
ax4.grid(True, axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

print("Budget sensitivity analysis and scenario chart are complete.")

In [ ]:
print("\n" + "=" * 70)
print("COMPREHENSIVE INVESTMENT ANALYSIS SUMMARY")
print("=" * 70)

print("\nOptimal portfolio summary:")
print(f"- Total investment: ${total_cost:.2f}M of ${budget:.2f}M")
print(f"- Budget utilization: {total_cost / budget * 100:.1f}%")
print(f"- Expected portfolio NPV: ${portfolio_npv:.2f}M")
print(f"- Number of selected projects: {len(selected_projects)}")

print("\nSelected project details:")
for index, project in enumerate(selected_projects, start=1):
    standalone_expected_npv = sum(
        scenario.probability
        * calculate_npv(
            project, scenario, discount_rate, planning_horizon
        )
        for scenario in scenarios
    )
    npv_to_investment_ratio = standalone_expected_npv / project.cost
    print(f"{index}. {project.name}")
    print(f"   - Initial investment: ${project.cost:.2f}M")
    print(f"   - Standalone expected NPV: ${standalone_expected_npv:.2f}M")
    print(
        "   - Standalone NPV / investment: "
        f"{npv_to_investment_ratio:.2%}"
    )

print("\nModel interpretation:")
print(
    "- The selected portfolio maximizes probability-weighted NPV "
    "subject to the stated budget and modeling assumptions."
)
print(
    "- Synergy adds value only when both projects in a compatible pair "
    "are selected."
)
print(
    "- Scenario NPVs show how the baseline portfolio performs under "
    "each modeled market outcome."
)
print(
    "- Optimality applies to this mathematical model and its specified "
    "input data."
)

print("\nMathematical model summary:")
print(f"- Binary project-selection variables: {len(projects)}")
print(f"- Binary synergy variables: {len(synergies)}")
print(
    f"- Annual activation variables: "
    f"{len(projects) * planning_horizon}"
)
print(f"- Budget constraint: total investment at most ${budget:.2f}M")
print(f"- Scenarios in expected NPV: {len(scenarios)}")
print(
    f"- Planning horizon: {planning_horizon} years; "
    f"discount rate: {discount_rate:.0%}"
)
print("=" * 70)